In [1]:
from rabi_freq import rabi_frequency_Rb85_hfs, rabi_frequency_Rb85_fs
import numpy as np
import matplotlib.pyplot as plt

In [39]:
AtomRadius = 250e-6

# Beam Parameters
BeamSizeMultiplier = 2
PowerUV = 0.2
PowerBlue = 0.35
WaistUV = BeamSizeMultiplier * AtomRadius
WaistBlue = BeamSizeMultiplier * AtomRadius
ground_state = (5, 0, 0.5, 0.5)
excited_state = (5, 1, 1.5, 1.5)
rydberg_r_state = (55, 0, 0.5)
rydberg_f_state = (54, 1, 1.5)

OmegaB = rabi_frequency_Rb85_fs(excited_state, rydberg_r_state, -1, PowerBlue, WaistBlue)["Omega_MHz"]

OmegaUV = rabi_frequency_Rb85_fs(ground_state, rydberg_f_state, 1, PowerUV, WaistUV)["Omega_MHz"]

In [40]:
print(f"Rabi frequency for blue beam: {OmegaB:.5f} MHz")
print(f"Rabi frequency for UV beam: {OmegaUV:.2f} MHz")

Rabi frequency for blue beam: 2.30054 MHz
Rabi frequency for UV beam: 0.43 MHz


In [41]:
from arc import Rubidium85
from scipy.constants import physical_constants

atom = Rubidium85()

ea0 = physical_constants["atomic unit of electric dipole moment"][0]  # C m
debye = 3.33564e-30  # C m

# Example: 5P_3/2, mJ=+1/2 -> 55S_1/2, mJ=+1/2, pi light
d_blue = atom.getDipoleMatrixElement(
    excited_state[0], excited_state[1], excited_state[2], excited_state[3],
    rydberg_r_state[0], rydberg_r_state[1], rydberg_r_state[2], 0.5,
    -1                  # q: sigma-
)
d_UV = atom.getDipoleMatrixElement(
    ground_state[0], ground_state[1], ground_state[2], ground_state[3],
    rydberg_f_state[0], rydberg_f_state[1], rydberg_f_state[2], 1.5,
    1                   # q: sigma+
)

print(f"Blue transition dipole moment: {d_blue:.5e} ea0")
print(f"UV transition dipole moment: {d_UV:.5e} ea0")

Blue transition dipole moment: -6.93809e-03 ea0
UV transition dipole moment: 1.70762e-03 ea0


In [6]:
d_au = atom.getDipoleMatrixElement(ground_state[0], ground_state[1], ground_state[2], ground_state[3],
                                   excited_state[0], excited_state[1], excited_state[2], excited_state[3], 1)
d_Cm = d_au * ea0
d_D = d_Cm / debye
print("d =", d_au, "ea0")
print("d =", d_Cm, "C m")
print("d =", d_D, "Debye")

d = 2.98915 ea0
d = 2.53430691389735e-29 C m
d = 7.59766315878617 Debye


In [32]:
hbar = physical_constants["reduced Planck constant"][0]  # J s
epsilon_0 = physical_constants["electric constant"][0]  # F/m
c = physical_constants["speed of light in vacuum"][0]  # m/s
def G(d, omega, w, L,N, wa):
    return d * np.sqrt((omega/ (hbar * epsilon_0 * np.pi * w**2 * L))*((N)/(1+ 4*(wa**2/w**2))))

def G_fromV(d, omega, w, V, N,wa):
    return d * np.sqrt((omega/ (2 *hbar * epsilon_0 * V))*((N)/(1+ 4*(wa**2/w**2))))

lambda_opt = 780.24 * 1e-9  # m
omega_opt = 2 * np.pi * c / lambda_opt  # rad/s
L = 2.54 * 1e-2  # m
N  = 1500
V = 0.083 * (1e-3)**3  # m^3
w = 55 * 1e-6  # m
wa = 70 * 1e-6  # m

G_eff = G_fromV(d_Cm, omega_opt, w, V, N, wa) / (2*np.pi*10**6)  # Convert to MHz
print("G_eff =", G_eff) 
print("N_eff =", N/(1+ 4*(wa**2/w**2)))

G_eff = 7.128744598253951
N_eff = 200.5524861878453
